In [15]:
import pandas as pd
import numpy as np
import time
import psutil
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

LOAD DATASET

In [16]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Dataset Shape:", df.shape)

df.head()

Saving HuGaDB_v2_various_01_00.csv to HuGaDB_v2_various_01_00 (1).csv
Dataset Shape: (2435, 40)


,Unnamed: 0,accelerometer_right_foot_x,accelerometer_right_foot_y,accelerometer_right_foot_z,gyroscope_right_foot_x,gyroscope_right_foot_y,gyroscope_right_foot_z,accelerometer_right_shin_x,accelerometer_right_shin_y,accelerometer_right_shin_z,...,gyroscope_left_shin_z,accelerometer_left_thigh_x,accelerometer_left_thigh_y,accelerometer_left_thigh_z,gyroscope_left_thigh_x,gyroscope_left_thigh_y,gyroscope_left_thigh_z,EMG_right,EMG_left,activity
0,0,-7868,-1336,13400,-28,26,-24,-14920,-240,-6800,...,18,-5696,2088,16016,26,-3,9,120,129,sitting
1,1,-7956,-1384,13468,-20,28,-22,-14824,-216,-6928,...,12,-5608,2224,16184,20,-3,7,120,128,sitting
2,2,-7892,-1348,13492,-18,24,-26,-14816,-144,-6808,...,13,-5512,2168,16216,21,-7,9,120,129,sitting
3,3,-7856,-1352,13504,-16,23,-21,-14984,-56,-7200,...,17,-5552,2184,16048,20,-3,4,121,128,sitting
4,4,-7896,-1356,13440,-13,26,-21,-14944,-280,-6920,...,16,-5536,2032,16040,21,-9,7,121,128,sitting


INFO ABOUT DATASET

In [17]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nActivities:")
print(df["activity"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2435 entries, 0 to 2434
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Unnamed: 0                   2435 non-null   int64 
 1   accelerometer_right_foot_x   2435 non-null   int64 
 2   accelerometer_right_foot_y   2435 non-null   int64 
 3   accelerometer_right_foot_z   2435 non-null   int64 
 4   gyroscope_right_foot_x       2435 non-null   int64 
 5   gyroscope_right_foot_y       2435 non-null   int64 
 6   gyroscope_right_foot_z       2435 non-null   int64 
 7   accelerometer_right_shin_x   2435 non-null   int64 
 8   accelerometer_right_shin_y   2435 non-null   int64 
 9   accelerometer_right_shin_z   2435 non-null   int64 
 10  gyroscope_right_shin_x       2435 non-null   int64 
 11  gyroscope_right_shin_y       2435 non-null   int64 
 12  gyroscope_right_shin_z       2435 non-null   int64 
 13  accelerometer_right_thigh_x  2435

SYSTEM INFORMATION

In [5]:
print("="*50)
print("SYSTEM INFORMATION")
print("="*50)

print("CPU Cores:", psutil.cpu_count())

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total/(1024**3):.2f} GB")
print(f"Current RAM Usage: {ram.percent}%")

SYSTEM INFORMATION
CPU Cores: 2
Total RAM: 12.67 GB
Current RAM Usage: 10.0%


PREPROCESSING

In [18]:
df = df.drop("Unnamed: 0", axis=1)

X = df.drop("activity", axis=1)

y = df["activity"]

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (2435, 38)
Target Shape: (2435,)


TRAIN TEST SPLIT

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1948, 38)
Test Shape: (487, 38)


Model Profiling Function

In [20]:
results = []

def profile_model(model, model_name):

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    ram_before = psutil.virtual_memory().used / (1024**3)

    train_start = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - train_start

    ram_after = psutil.virtual_memory().used / (1024**3)

    infer_start = time.time()

    y_pred = model.predict(X_test)

    inference_time = time.time() - infer_start

    accuracy = accuracy_score(y_test, y_pred)

    model_file = f"{model_name}.pkl"

    joblib.dump(model, model_file)

    model_size = os.path.getsize(model_file)/(1024*1024)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Training Time: {training_time:.4f} sec")
    print(f"Inference Time: {inference_time:.6f} sec")
    print(f"RAM Increase: {(ram_after-ram_before):.4f} GB")
    print(f"Model Size: {model_size:.4f} MB")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Training_Time_sec": training_time,
        "Inference_Time_sec": inference_time,
        "RAM_Used_GB": ram_after-ram_before,
        "Model_Size_MB": model_size
    })

    return model

Logistic Regression

In [22]:
lr = LogisticRegression(max_iter=3000)

profile_model(
    lr,
    "Logistic_Regression"
)


Logistic_Regression
Accuracy: 0.9836
Training Time: 9.4427 sec
Inference Time: 0.002036 sec
RAM Increase: 0.0195 GB
Model Size: 0.0032 MB


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=3000)

SVM

In [10]:
svm = SVC()

profile_model(
    svm,
    "SVM"
)


SVM
Accuracy: 0.8953
Training Time: 0.0730 sec
Inference Time: 0.031829 sec
RAM Increase: 0.0000 GB
Model Size: 0.2643 MB


SVC()

RANDOM FORESTS

In [11]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

profile_model(
    rf,
    "Random_Forest"
)


Random_Forest
Accuracy: 0.9836
Training Time: 0.6809 sec
Inference Time: 0.013057 sec
RAM Increase: -0.0003 GB
Model Size: 0.7719 MB


RandomForestClassifier(random_state=42)

COMPARISON TABLE

In [12]:
comparison_df = pd.DataFrame(results)

comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Training_Time_sec,Inference_Time_sec,RAM_Used_GB,Model_Size_MB
0,Logistic_Regression,0.985626,1.132479,0.002649,-0.003155,0.003211
2,Random_Forest,0.983573,0.680908,0.013057,-0.000286,0.771874
1,SVM,0.895277,0.072991,0.031829,0.000000,0.264312


MODEL ANALYSIS

In [13]:
best_accuracy = comparison_df.loc[
    comparison_df["Accuracy"].idxmax()
]

fastest_model = comparison_df.loc[
    comparison_df["Inference_Time_sec"].idxmin()
]

smallest_model = comparison_df.loc[
    comparison_df["Model_Size_MB"].idxmin()
]

print("Highest Accuracy:")
print(best_accuracy)

print("\nFastest Inference:")
print(fastest_model)

print("\nSmallest Model:")
print(smallest_model)

Highest Accuracy:
Model                 Logistic_Regression
Accuracy                         0.985626
Training_Time_sec                1.132479
Inference_Time_sec               0.002649
RAM_Used_GB                     -0.003155
Model_Size_MB                    0.003211
Name: 0, dtype: object

Fastest Inference:
Model                 Logistic_Regression
Accuracy                         0.985626
Training_Time_sec                1.132479
Inference_Time_sec               0.002649
RAM_Used_GB                     -0.003155
Model_Size_MB                    0.003211
Name: 0, dtype: object

Smallest Model:
Model                 Logistic_Regression
Accuracy                         0.985626
Training_Time_sec                1.132479
Inference_Time_sec               0.002649
RAM_Used_GB                     -0.003155
Model_Size_MB                    0.003211
Name: 0, dtype: object
